Start time of execution

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymongo
import yaml
from time import time
import numpy as np #for tensor operations
from utils import sliding_window_pd, filter_instances, encode_labels, check_missing_values
from utils_visual import (
    plot_filtered_vs_raw_signal,
    plot_feature_boxplots_by_class,
    plot_scatter_pca,
    plot_window_counts_by_gesture_user
)
from utils_features import extract_all_candidates

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from tensorflow.keras import layers, models

**Load Configuation**

In [ ]:
config_path = os.path.join(os.getcwd(), "config.yml")
with open(config_path, "r") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

In [ ]:
time_start = time()


In [ ]:
client = pymongo.MongoClient(config["client"])
db = client[config["db"]]
collection = db[config["col"]]


In [ ]:
found_keys = collection.distinct("_id")
print("Total number of documents in the collection:", len(found_keys))

In [ ]:
documents = collection.find({"_id": {"$in": found_keys}})
rows = []
for doc in documents:
    n_samples = len(doc["data"]["acc_x"])
    for i in range(n_samples):
        rows.append({
            "acc_x": float(doc["data"]["acc_x"][i]),
            "acc_y": float(doc["data"]["acc_y"][i]),
            "acc_z": float(doc["data"]["acc_z"][i]),
            "gyr_x": float(doc["data"]["gyr_x"][i]),
            "gyr_y": float(doc["data"]["gyr_y"][i]),
            "gyr_z": float(doc["data"]["gyr_z"][i]),
            "gesture_id": doc["gesture_id"],

            "user": doc["user"],
        })

dataframe = pd.DataFrame(rows)
# Should be float/int
print(dataframe[["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z", "gesture_id","user"]].dtypes)
print(dataframe[:5])

## Data Exploration
This section will explore the dataset to understand its structure, identify any missing values, and analyze the distribution of the target variable.

### Step 1: Barplot of each recording

In [ ]:
dict = {}
for key,value in dataframe.groupby(["gesture_id", "user"]):
    new_key = f"{key[0]}_{key[1]}"
    dict[new_key] = len(value) / 100 # sampling rate is 100Hz, so we divide by 100 to get the count in seconds
plt.figure(figsize=(12, 6))
sns.barplot(x=list(dict.keys()), y=list(dict.values()))
plt.xticks(rotation=90)
plt.xlabel("Gesture ID and User")
plt.ylabel("Count (in seconds)")
plt.title("Count of Each Gesture ID and User")
plt.show()


### Step 2: Checking for Missing Values
We check for missing values in the dataset using the `check_missing_values` utility function to ensure data integrity before filtering and windowing. 

Additionally, we examine the data for outliers. In IMU sensor signal processing, outliers (extreme acceleration or angular velocity values) are not removed or clipped, because they represent the natural peak force and physical velocity of the gestures. Discarding them would eliminate crucial movement signatures that are essential for deep learning models like CNNs.

In [ ]:
check_missing_values(dataframe)

### Step 3: Filter Signals

In [ ]:
list_of_signals = [value for key,value in dataframe.groupby(["gesture_id","user"])]
filtered_list_of_signals = filter_instances(
        instances_list = list_of_signals,
        order=config["filter"]["order"],
        filter_type=config["filter"]["type"],
        wn = config["filter"]["wn"]
    )

In [ ]:
plot_filtered_vs_raw_signal(
    raw_signals = list_of_signals,
    filtered_signals = filtered_list_of_signals,
    idx = 0,
    n_start = 0,
    n_end = 100
)

### Step 4: Windowing

In [ ]:
# Windowing
windows = []
for signal in filtered_list_of_signals:
    temp_windows = sliding_window_pd(
        df=signal,
        ws=config["sliding_window"]["ws"],
        overlap=config["sliding_window"]["overlap"],
        w_type=config["sliding_window"]["w_type"],
    )
    for window in temp_windows:
        windows.append(window)

print(f"Total number of windows: {len(windows)}")
print(f"Shape of first window: {windows[0].shape}")

### Step 5: Windows Barplot
Windowing Barploting in respect to class distribution and user distribution. That helps us to identify if a user and a specific activity is overrepresented in the dataset.

In [ ]:
plot_window_counts_by_gesture_user(windows)


### Convert windows to tensor format
This step will convert the windowed data into a format suitable for machine learning models (Deep Neural Networks), such as tensors.

In [ ]:
train_windows, test_windows = train_test_split(windows, test_size=0.2, random_state=42, stratify=[f"{window['gesture_id'].iloc[0]}_{window['user'].iloc[0]}" for window in windows])
sensor_columns = ["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z"]
X_train = np.stack([window[sensor_columns].to_numpy() for window in train_windows])
y_train = np.array([window["gesture_id"].iloc[0] for window in train_windows])
y_train_encoded, le = encode_labels(y_train)
X_test = np.stack([window[sensor_columns].to_numpy() for window in test_windows])
y_test = np.array([window["gesture_id"].iloc[0] for window in test_windows])
y_test_encoded, _ = encode_labels(y_test)
X_train.shape, y_train_encoded.shape

### Perform Scaling in each tensor
This step will apply scaling to the tensor data, which is crucial for improving the performance of many machine learning algorithms.

In [ ]:
from sklearn.preprocessing import StandardScaler

n_train, window_size, n_channels = X_train.shape

scaler = StandardScaler()

X_train_2d = X_train.reshape(-1, n_channels)
X_test_2d = X_test.reshape(-1, n_channels)

X_train_scaled = scaler.fit_transform(X_train_2d).reshape(X_train.shape)
X_test_scaled = scaler.transform(X_test_2d).reshape(X_test.shape)

### Apply 1D CNN
This step will involve applying a 1D Convolutional Neural Network (CNN) to the scaled tensor data for classification tasks. The CNN will learn to extract relevant features from the time series data and make predictions based on those features.

We tried the CNN model to be as much simple as possible, to be able to compare it with the results of the SVM and Random Forest models. We used a simple architecture with one convolutional layer followed by a max-pooling layer and a fully connected layer. The model was trained for a few epochs to demonstrate the process.

In [ ]:
num_classes = len(np.unique(y_train_encoded))
model = models.Sequential([
    layers.Input(shape=(config['sliding_window']['ws'], 6)),  # 6 channels: acc_x, acc_y, acc_z, gyr_x, gyr_y, gyr_z

    layers.Conv1D(64, kernel_size=5, padding="same"),
    layers.BatchNormalization(),
    layers.ReLU(),

    layers.GlobalAveragePooling1D(),

    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train_encoded,
    validation_split=0.2,
    epochs=50,
    batch_size=32
)

In [ ]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test_encoded)
print(f"Test Accuracy: {test_acc:.4f}")

y_pred = model.predict(X_test_scaled).argmax(axis=1)

print(
    classification_report(
        y_test_encoded,
        y_pred,
        target_names=[str(x) for x in le.classes_]
    )
)

from sklearn.metrics import confusion_matrix
from utils_visual import plot_confusion_matrix
cm = confusion_matrix(y_test_encoded, y_pred)
plot_confusion_matrix(
    cm=cm,
    labels=list(le.classes_),
    output_dir="figures/evaluation/cnn",
    filename="confusion_cnn_stratified_split.png",
    save_fig=True
)



In [ ]:
time_end = time()
elapsed_time = time_end - time_start
print(f"Total execution time: {elapsed_time:.2f} seconds")